# Week 7: Dirty cups counter (object recognition)

## The technique: object recognition

Object recognition answers three questions about a photo: **what** is in it, **where** is it, and **how many** are there.

- **Input:** a photo.
- **Output:** for every object a **box** (where), a **label** (what) and a **confidence** (how sure the model is, from 0 to 1).
- **Used for:** counting things, finding things, checking whether something is there.

## My MVP

**User:** someone in a student house who wants to know how many dirty cups are left in the kitchen.

**MVP:** a photo in, an answer out. You take a photo of the counter or the sink, and the app says: *"3 cups to wash"*.

**How:** a pre-trained model finds all objects in the photo. I keep only the boxes with the label `cup` and count them.

## Steps in this notebook

1. Setup
2. Load my photos
3. Data processing: from a phone photo to the model
4. The model
5. The MVP: photo in, answer out
6. Ground truth
7. Evaluation (precision, recall, count error, other thresholds, other model)
8. Where it fails
9. My results
10. Where it fails, explained
11. Who could it fail, and who could be harmed?

## 1. Setup

Load the libraries: `torch` and `torchvision` for the model, `PIL` to open the photos, `matplotlib` to draw the boxes.

In [ ]:
import os, csv
import torch, torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
import matplotlib.patches as patches

## 2. Load the photos

13 photos of mugs, taken with my phone at home. I made them hard on purpose: 1 to 3 mugs, upright, upside down, lying on their side and seen from the top, in evening light. No people.

To run it: upload `photos_for_colab.zip` with the folder icon on the left. The cell below unpacks it.

In [ ]:
# If you uploaded photos_for_colab.zip, this unpacks it (photos/ + ground_truth.csv)
if os.path.exists("photos_for_colab.zip"):
    import zipfile
    zipfile.ZipFile("photos_for_colab.zip").extractall(".")

PHOTO_DIR = "photos"
files = sorted(f for f in os.listdir(PHOTO_DIR) if f.lower().endswith((".jpg", ".jpeg", ".png")))
print(len(files), "photos:")
for f in files:
    print(" ", f)

## 3. Data processing: from a phone photo to the model

Garbage in, garbage out: a model can only be as good as the pixels you feed it. Phone photos all differ (light, angle, distance, size), and the model expects one format. So every photo goes through the same steps:

1. **Open** the photo.
2. **Straighten:** turn it the right way. A phone stores the rotation separately, so we apply it.
3. **Keep the colour (RGB, 3 numbers per pixel).** We do not make it grey, because the model learned from colour photos.
4. **Resize:** longest side 1024 pixels. Smaller is faster, but we keep the aspect ratio, so cups are not squeezed.
5. **Normalise:** pixel values 0 to 255 become 0 to 1. Nothing visible changes, only the numbers. We use the same recipe the model was trained with.

Now the image is an array of numbers (a tensor): channels × height × width. This is what the model sees.

In [ ]:
MAX_SIDE = 1024
to_tensor = FasterRCNN_ResNet50_FPN_Weights.DEFAULT.transforms()

def load_photo(path):
    img = Image.open(path)
    img = ImageOps.exif_transpose(img)   # turn the right way
    img = img.convert("RGB")             # 3 colour channels
    img.thumbnail((MAX_SIDE, MAX_SIDE))  # make smaller
    return img

img = load_photo(os.path.join(PHOTO_DIR, files[0]))
print("size:", img.size)
print("tensor shape:", to_tensor(img).shape)  # [channels, height, width]

## 4. The model

A pre-trained **Faster R-CNN** model. It already learned 91 kinds of objects (the COCO dataset), and `cup` is one of them. I do not train it myself.

`detect` gives every object the model finds (label, confidence, box). `count_cups` counts only the boxes with the label `cup` above the threshold.

In [ ]:
weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn(weights=weights).eval()
LABELS = weights.meta["categories"]

def detect(img):
    with torch.no_grad():
        out = model([to_tensor(img)])[0]
    return [(LABELS[l], float(s), b.tolist()) for l, s, b in zip(out["labels"], out["scores"], out["boxes"])]

def count_cups(detections, threshold):
    return sum(1 for label, score, box in detections if label == "cup" and score >= threshold)

## 5. The MVP: photo in, answer out

`THRESHOLD` is the minimum confidence. A box with a lower confidence is not counted.
Green boxes are cups. Grey boxes are other objects (to see what else the model finds).

In [ ]:
THRESHOLD = 0.5
results = {}

for f in files:
    img = load_photo(os.path.join(PHOTO_DIR, f))
    dets = detect(img)
    results[f] = dets
    n = count_cups(dets, THRESHOLD)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(img); ax.axis("off")
    nr = 0
    for label, score, (x1, y1, x2, y2) in dets:
        if score < THRESHOLD:
            continue
        colour = "lime" if label == "cup" else "grey"
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color=colour, linewidth=2))
        if label == "cup":
            nr += 1
            ax.text(x1, y1, f"{nr} {score:.2f}", color="black", backgroundcolor="lime", fontsize=8)
        else:
            ax.text(x1, y1, f"{label} {score:.2f}", color="white", backgroundcolor="grey", fontsize=7)
    ax.set_title(f"{f}: {n} cups to wash")
    plt.show()

## 6. Ground truth

The right answers, written down **by me, before I trust the model**.

`ground_truth.csv` (in `photos_for_colab.zip`). I counted the mugs in every photo myself before I ran the model. After the model ran, I judged every green box.

| column | what |
|---|---|
| `file` | the file name of the photo |
| `true_count` | how many cups are really in the photo (I counted them myself) |
| `correct_boxes` | look at the green boxes above: how many are really on a cup? (a box on something else, or two boxes on one cup, is wrong) |

In [ ]:
gt = {}
with open("ground_truth.csv") as fh:
    for row in csv.DictReader(fh):
        c = row["correct_boxes"].strip()
        gt[row["file"]] = {"true": int(row["true_count"]), "correct": int(c) if c else None}
missing = [f for f in files if f not in gt]
print("photos without ground truth:", missing or "none")
not_judged = [f for f in gt if gt[f]["correct"] is None]
if not_judged:
    print("Boxes not judged yet:", not_judged)
    print("Fill in correct_boxes in ground_truth.csv for these photos.")

## 7. Evaluation

Three metrics:

- **Precision** = correct boxes / all boxes the model gave. *When the model says "cup", how often is it right?*
- **Recall** = correct boxes / all real cups. *How many of the real cups did the model find?*
- **Count error** = |model count − true count|, averaged over the photos. *This is what the user cares about: is the number right?*

Why these: the user only gets a number, so the count error matters most. Precision and recall show **why** the count is wrong: too many boxes (low precision) or missed cups (low recall).

In [ ]:
total_pred = total_correct = total_true = 0
rows = []
for f in files:
    if f not in gt:
        continue
    pred = count_cups(results[f], THRESHOLD)
    t, c = gt[f]["true"], gt[f]["correct"]
    total_pred += pred; total_true += t
    total_correct += c if c is not None else 0
    rows.append((f, t, pred, c, abs(pred - t)))

print(f"{'photo':30} true  model  correct  error")
for f, t, p, c, e in rows:
    print(f"{f:30} {t:4}  {p:5}  {str(c if c is not None else '-'):>7}  {e:5}")

precision = total_correct / total_pred if total_pred else 0
recall = total_correct / total_true if total_true else 0
count_error = sum(r[4] for r in rows) / len(rows)
exact = sum(1 for r in rows if r[4] == 0)
print()
print(f"Threshold     : {THRESHOLD}")
if any(gt[f]["correct"] is None for f in gt):
    print("Precision     : fill in correct_boxes first")
    print("Recall        : fill in correct_boxes first")
else:
    print(f"Precision     : {precision:.2f}")
    print(f"Recall        : {recall:.2f}")
print(f"Count error   : {count_error:.2f} cups per photo")
print(f"Exactly right : {exact} of {len(rows)} photos")

### Try other thresholds

A low threshold gives more boxes (more found, but more wrong ones). A high threshold gives fewer boxes (fewer wrong ones, but more missed). Which one gives the lowest count error?

In [ ]:
for th in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    errs = [abs(count_cups(results[f], th) - gt[f]["true"]) for f in files if f in gt]
    print(f"threshold {th:.1f}: count error {sum(errs)/len(errs):.2f}, exactly right {sum(e == 0 for e in errs)} of {len(errs)}")

### Try another model

Does a different model do better? Here I compare with a smaller, faster model (Faster R-CNN MobileNet). Same photos, same ground truth, same threshold.

In [ ]:
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn, FasterRCNN_MobileNet_V3_Large_FPN_Weights
small_model = fasterrcnn_mobilenet_v3_large_fpn(weights=FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT).eval()

errs = []
for f in files:
    if f not in gt:
        continue
    img = load_photo(os.path.join(PHOTO_DIR, f))
    with torch.no_grad():
        out = small_model([to_tensor(img)])[0]
    n = sum(1 for l, s in zip(out["labels"], out["scores"]) if LABELS[l] == "cup" and s >= THRESHOLD)
    errs.append(abs(n - gt[f]["true"]))
print(f"Big model   : count error {count_error:.2f}")
print(f"Small model : count error {sum(errs)/len(errs):.2f}")

## 8. Where it fails

The photos with the biggest error. For each one, look: is it the model's mistake, or the ground truth's?

In [ ]:
worst = [r for r in sorted(rows, key=lambda r: r[4], reverse=True) if r[4] > 0]
for f, t, p, c, e in worst:
    print(f"{f}: true {t}, model {p}, error {e}")

## 9. My results

13 photos, 19 mugs (1 to 3 per photo), taken with my phone at home in evening light.

| Metric | Result |
|---|---|
| Precision | **1.00**: all 14 cup boxes were on a real mug |
| Recall | **0.74**: 14 of the 19 mugs found |
| Count error | **0.38** cups per photo |
| Exactly right | **8 of 13** photos |

**Thresholds:** 0.3 to 0.7 all give a count error of 0.38. At 0.8 it gets worse (0.54). The threshold is not the problem: the model never puts a cup box on something else, but some mugs do not get the label cup.

**Other model:** the smaller Faster R-CNN MobileNet also has a count error of 0.38.

## 10. Where it fails

| Photo | True | Model | What is in the photo |
|---|---|---|---|
| IMG_1963 | 1 | 0 | blue mug lying on its side, labelled **cell phone** (0.59) |
| IMG_1965 | 1 | 0 | white mug lying on its side, labelled **remote** (0.56) |
| IMG_1966 | 1 | 0 | white mug lying on its side, labelled **cell phone** (0.88) |
| IMG_1970 | 3 | 2 | 3 mugs, 2 lying down: the lying "uncle" mug is labelled **clock** (0.56) |
| IMG_1971 | 3 | 2 | 3 mugs, 2 lying down: the lying "uncle" mug is labelled **cell phone** (0.90) |

- Mugs standing (upright or upside down): **9 of 9** counted as cup
- Mugs lying on their side: **5 of 10** counted as cup

**What goes wrong:** the model does not miss the lying mugs. It finds them, but gives them the **wrong label**: cell phone, remote or clock. On its side a mug looks like a flat rectangle or a round face, and that is the shape of a phone, a remote or a clock. When the round opening faces the camera (IMG_1968), it does see a cup. The model learned cups from photos where cups stand on a table.

**Other wrong labels (grey boxes, they do not change the count):** the white spot inside the mug in IMG_1962 is called a donut (0.62), the lion mug in IMG_1973 also gets a toilet box (0.67), and in IMG_1961 the model sees a chair in the corner.

**Model or ground truth?** The model. The mugs are easy to count for a person, so my ground truth is right.

## 11. Who could it fail, and who could be harmed?

**Who could it fail?**
- People whose mugs lie on their side, like in a full sink. Half of those were not counted.
- People with cups that look different from a normal mug, because the model only knows cups from its training photos.

**Who could be harmed?**
- For a cup counter, the harm is small: a wrong count means a cup stays dirty.
- It gets serious if the same model is used for something bigger, like a canteen that counts returned cups to give back a deposit. Then someone who returns a cup lying on its side does not get their money back.

**What I would do:** show the boxes so people can check the count, and test with more photos of cups lying down before using it for anything important.

No people are in the photos (I cut a hand out of IMG_1969).